# Thảo luận và kết luận

## Kết quả đã có bằng chứng

Nhóm đã nối được pipeline M1–M2: từ metadata/version và corpus nguồn đến năm bảng
có kiểm tra cấu trúc. Snapshot nhỏ cho phép rà từng version, đối chiếu lỗi với source
và chạy lại parser trên cùng đầu vào. Các biểu đồ của Book lấy từ thống kê của chính
các lượt này; số đo của nhóm khác không được đưa vào kết quả.

## Các điểm còn hạn chế

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
data = json.loads((Path.cwd().parent / 'report_summary.json').read_text(encoding='utf-8'))
scope, m1, m2 = data['scope'], data['m1'], data['m2']
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False, 'axes.spines.right': False})

In [ ]:
missing_sources = scope['versions'] - m1['source_verified']
unresolved_s2 = scope['papers'] - m1['verified_reference_papers']
display(Markdown('**Bảng T.1. Phần việc còn lại trên snapshot, theo mức phụ thuộc.**'))
display(pd.DataFrame([
    (1, 'Nguồn', missing_sources, 'version', 'Xác minh hoặc tải lại đúng revision rồi audit'),
    (2, 'S2 chưa success/empty', unresolved_s2, 'paper', 'Truy vấn lại theo cooldown, kiểm tra phân trang và provenance'),
    (3, 'Parser partial', m2['status_counts'].get('partial', 0), 'version', 'Đối chiếu include, bibliography và citation diagnostics'),
    (4, 'Root/parser error', m2['status_counts'].get('error', 0), 'version', 'Chọn root khi source có bằng chứng, rồi parse lại'),
    (5, 'Benchmark M2 Colab cuối', 'Chưa có', 'lượt đo', 'Chạy notebook CPU-only trên cùng input và lưu output thật'),
], columns=['Thứ tự', 'Hạng mục', 'Số lượng', 'Đơn vị', 'Cách hoàn thiện']))
display(Markdown('Các hàng có thể liên quan cùng một paper; không cộng chúng thành tổng paper lỗi. '
                 '“Chưa có” nghĩa là chưa có phép đo Colab tương ứng.'))

## Phân tích lỗi và tác động

References thiếu làm giảm tập ứng viên cho bài toán matching về sau, ngay cả khi paper
đã có đủ TeX. Unresolved citation khiến việc liên kết nội dung với bibliography chưa
trọn vẹn; đó là lý do các version này được giữ partial. Một compilation root chưa
xác định được phải được xử lý riêng, vì chọn nhầm root có thể tạo một cây hợp lệ về
hình thức nhưng chứa sai tài liệu.

Các nhóm diagnostic có thể cùng xuất hiện trong một version. Biểu đồ dưới đếm version
có từng loại diagnostic, không khẳng định một nguyên nhân duy nhất cho mọi lỗi.

In [ ]:
diagnostics = pd.Series(m2['diagnostic_categories']).sort_values()
ax = diagnostics.plot.barh(color='#b57b36', xlabel='Version có diagnostic', ylabel='')
for i, count in enumerate(diagnostics):
    ax.text(count + .25, i, str(count), va='center')
ax.set_xlim(0, diagnostics.max() * 1.15)
plt.tight_layout(); plt.show()
display(Markdown('**Hình T.1. Các nhóm diagnostic trên snapshot.** Nguồn: tổng hợp manifest parser.'))

## Hiệu năng và giới hạn suy luận

M1 chịu ảnh hưởng của API, mạng, kích thước archive và thời gian chờ; M2 chủ yếu thao tác
trên nguồn cục bộ. Hai lượt đang có khác phạm vi và môi trường, nên không dùng chúng để
so tốc độ trực tiếp. Số đo mẫu cũng chưa đủ để bảo đảm thời gian hoàn tất 25.000 paper.

Snapshot được chọn theo archive sẵn có và giới hạn tải, không theo thiết kế lấy mẫu
ngẫu nhiên. Vì vậy tỷ lệ thành công, kích thước trung bình và mức giảm dedup mô tả tập
đã kiểm tra; chưa đại diện chắc chắn cho cả dải được giao.

## Kết luận

Kết quả hiện tại cho thấy pipeline có thể tạo đầu ra M1–M2 và phát hiện rõ các trường
hợp chưa xử lý xong. Để hoàn tất dữ liệu, nhóm cần khép phần nguồn/references còn thiếu,
xử lý diagnostics, chạy validation lại trên phạm vi M1 đã chốt và bổ sung benchmark
Colab. Những lần cập nhật tiếp tục dùng cùng repository source và URL Book.